In [2]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os

builder = SparkSession.builder \
    .appName("ETL Gold Layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
silver_path = "../delta_lake/silver/"

dim_artists = spark.read.format("delta").load(os.path.join(silver_path, "dim_artists"))
dim_albums = spark.read.format("delta").load(os.path.join(silver_path, "dim_albums"))
dim_genres = spark.read.format("delta").load(os.path.join(silver_path, "dim_genres"))
dim_tracks = spark.read.format("delta").load(os.path.join(silver_path, "dim_tracks"))
fact_tracks = spark.read.format("delta").load(os.path.join(silver_path, "fact_tracks"))

In [4]:
gold_path = "../delta_lake/gold/"

## Avg metrics per album

In [ ]:
from pyspark.sql.functions import avg

selected_metrics = [
    "popularity",
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence"
]

fact_with_album = fact_tracks.join(dim_tracks.select("track_id", "album_id"), on="track_id", how="inner")
fact_with_album = fact_with_album.join(dim_albums.select("album_id", "album_name", "artist_id"), on="album_id", how="inner")

# join dim_artists to get artist names
fact_with_album_artist = fact_with_album.join(dim_artists.select("artist_id", "artists"), on="artist_id", how="inner")

# aggregate by album name and artist
aggregated_by_album_artist = fact_with_album_artist.groupBy("album_name", "artists").agg(
    *[avg(c).alias(f"avg_{c}") for c in selected_metrics]
)

aggregated_by_album_artist.show(5, truncate=False)

+-------------------------------------+--------------------------+--------------+------------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+
|album_name                           |artists                   |avg_popularity|avg_danceability  |avg_energy         |avg_speechiness    |avg_acousticness   |avg_instrumentalness|avg_liveness       |avg_valence       |
+-------------------------------------+--------------------------+--------------+------------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+
|pov: you have a holly jolly christmas|Albert King               |0.0           |0.5439999999999999|0.46048484848484855|0.0669909090909091 |0.5851848484848484 |7.417727272727272E-5|0.17483333333333334|0.6198787878787879|
|Abra Sua Cabeça                      |Abayomy Afrobeat Orquestra|22.5          |0.696             |0.88650000000000

In [6]:
aggregated_by_album_artist.write.format("delta").mode("overwrite").save(gold_path + "aggregated_by_album_artist")

## Testing

In [7]:
fact_with_track = fact_tracks.join(dim_tracks, on="track_id", how="inner")

fact_with_track_selected = fact_with_track.select(
    dim_tracks["track_name"],
    *[col for col in fact_tracks.columns if col != "track_id"]
)

fact_with_track_selected.show(5, truncate=False)

+--------------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|track_name                |popularity|duration_ms|explicit|danceability|energy|loudness|tempo  |speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|
+--------------------------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|Comedy                    |73        |230666     |0       |0.676       |0.461 |-6.746  |87.917 |0.143      |0.0322      |1.01E-6         |0.358   |0.715  |0   |1  |4             |
|Ghost - Acoustic          |55        |149610     |0       |0.42        |0.166 |-17.235 |77.489 |0.0763     |0.924       |5.56E-6         |0.101   |0.267  |1   |1  |4             |
|To Begin Again            |57        |210826     |0       |0.438       |0.359 |-9.734  |76.332